## 1. Setup & Dependencies

**Installation (JupyterHub/Lab):**

```bash
# Create Python 3.10 environment
python3.10 -m venv .venv
source .venv/bin/activate

# Install main project dependencies
pip install -e ".[gpu-cuda12]"

# Install HOMR (in /home/jovyan/homr/)
cd ~
git clone https://github.com/liebharc/homr.git
cd homr
pip install poetry
python3.10 -m venv .venv
source .venv/bin/activate
poetry install --only main,gpu
cd ~/notes2tone

# Register Jupyter kernel
pip install ipykernel
python -m ipykernel install --user --name notes2tone --display-name "notes2tone"
```

In [ ]:
# Check GPU availability
import subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, timeout=5)
print(f"GPU: {gpu.stdout.strip()}" if gpu.returncode == 0 else "GPU: Not available")

## 2. HuggingFace Authentication *(Token required)*

In [ ]:
import os
from getpass import getpass

if 'HF_TOKEN' not in os.environ:
    print("HuggingFace Token Required")
    print("Token URL: https://huggingface.co/settings/tokens")
    print("Dataset access: https://huggingface.co/datasets/PRAIG/SMB")
    hf_token = getpass("Enter HF_TOKEN: ")
    os.environ['HF_TOKEN'] = hf_token
    print("Token configured successfully")
else:
    print("HF_TOKEN already configured")

## 3. Import Benchmark Framework

In [ ]:
from benchmarks.datasets import SMBDataset
from benchmarks.models import OemerModel, HomrModel
from benchmarks.benchmark import BenchmarkRunner
from benchmarks.eval import omr_ned

print("Benchmark framework loaded")

## 4. Load PRAIG/SMB Dataset

In [ ]:
NUM_SAMPLES = 10  # Set to None for full dataset

dataset = SMBDataset(
    split="test",
    limit=NUM_SAMPLES,
    token=os.environ.get('HF_TOKEN')
)

print(f"Dataset loaded: {len(dataset)} samples")

## 5. Initialize Models

In [ ]:
import sys
import shutil
from pathlib import Path

# Find OeMeR - check venv first, then PATH
oemer_path = Path(sys.prefix) / "bin" / "oemer"
if not oemer_path.exists():
    oemer_path = shutil.which("oemer")
else:
    oemer_path = str(oemer_path)
    
print(f"{'found' if oemer_path else 'not found'} OeMeR: {oemer_path or 'not found'}")

# Find HOMR - check common locations
homr_repo = Path.home() / "homr"
if not homr_repo.exists():
    homr_repo = Path.cwd() / "homr"
if not homr_repo.exists():
    homr_repo = None
    
print(f"{'found' if homr_repo else 'not found'} HOMR: {homr_repo or 'not found'}")

In [ ]:
models = []

# OeMeR
if oemer_path:
    try:
        oemer = OemerModel(
            oemer_path=oemer_path,
            disable_deskew=True,
            save_cache=False,
            use_tf=False
        )
        models.append(oemer)
        print(f"{oemer.name}")
    except Exception as e:
        print(f"OeMeR: {e}")
else:
    print("OeMeR: executable not found")

# HOMR
if homr_repo:
    try:
        homr = HomrModel(
            homr_path="homr",
            homr_dir=str(homr_repo),
            force_cpu=False
        )
        models.append(homr)
        print(f"{homr.name}")
    except Exception as e:
        print(f"HOMR: {e}")
else:
    print("HOMR: repository not found")
print(f"\n{len(models)} models ready for benchmarking")

## 6. Configure & Run Benchmark

In [ ]:
from pathlib import Path

SAVE_PREDICTIONS = True

output_dir = Path("benchmark_results")
output_dir.mkdir(exist_ok=True)

runner = BenchmarkRunner(
    dataset=dataset,
    output_dir=output_dir,
    save_predictions=SAVE_PREDICTIONS
)

print(f"Benchmark configured")
print(f"  Models: {len(models)}")
print(f"  Samples: {len(dataset)}")
print(f"  Output: {output_dir}")

## 7. Run Benchmark

In [ ]:
import time
from datetime import datetime

# Store all results
all_results = []

print(f"Starting benchmark for {len(models)} models")
print(f"Dataset: {len(dataset)} samples")
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

# Loop through all models
for i, model in enumerate(models, 1):
    print(f"\n[{i}/{len(models)}] Benchmarking: {model.name}")
    print("-" * 70)
    
    start_time = time.time()
    
    try:
        results = runner.evaluate_model(model)
        elapsed = time.time() - start_time
        
        results['elapsed_time'] = elapsed
        results['avg_time_per_sample'] = elapsed / len(dataset)
        all_results.append(results)
        
        m = results['metrics']
        print(f"Completed in {elapsed/60:.1f} min ({elapsed/len(dataset):.1f} sec/sample)")
        print(f"  Mean NED: {m['mean_ned']:.4f}")
        print(f"  Errors: {m['num_errors']}/{m['total_samples']} ({m['error_rate']*100:.1f}%)")
        
    except Exception as e:
        print(f"Failed: {e}")
        all_results.append({
            'model_name': model.name,
            'error': str(e),
            'elapsed_time': time.time() - start_time
        })

print("\n" + "=" * 70)
print(f"Benchmark complete: {len(all_results)} models evaluated")
print("=" * 70)

## 8. Comparison Summary

In [ ]:
# Display comparison table
print("\n" + "=" * 90)
print(f"{'Model':<15} | {'Mean NED':<10} | {'Median':<10} | {'Errors':<10} | {'Time (min)':<10}")
print("=" * 90)

for result in all_results:
    if 'error' in result:
        print(f"{result['model_name']:<15} | {'FAILED':<10} | {'-':<10} | {'-':<10} | {result['elapsed_time']/60:.1f}")
    else:
        m = result['metrics']
        print(f"{result['model_name']:<15} | "
              f"{m['mean_ned']:<10.4f} | "
              f"{m['median_ned']:<10.4f} | "
              f"{m['num_errors']}/{m['total_samples']:<6} | "
              f"{result['elapsed_time']/60:<10.1f}")

print("=" * 90)

# Find best model
successful_results = [r for r in all_results if 'error' not in r]
if successful_results:
    best = min(successful_results, key=lambda x: x['metrics']['mean_ned'])
    print(f"\nBest Model: {best['model_name']} (Mean NED: {best['metrics']['mean_ned']:.4f})")
else:
    print("\nNo models completed successfully")

## 9. Save Results

In [ ]:
from datetime import datetime
import json

# Save individual results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

for result in all_results:
    if 'error' not in result:
        filename = f"{result['model_name']}_results_{timestamp}.json"
        runner.save_results(result, filename)
        print(f"Saved: {filename}")

# Save comparison summary
comparison = {
    'timestamp': timestamp,
    'dataset': 'PRAIG/SMB',
    'num_samples': len(dataset),
    'models': []
}

for result in all_results:
    if 'error' in result:
        comparison['models'].append({
            'name': result['model_name'],
            'status': 'failed',
            'error': result['error']
        })
    else:
        comparison['models'].append({
            'name': result['model_name'],
            'mean_ned': result['metrics']['mean_ned'],
            'median_ned': result['metrics']['median_ned'],
            'error_rate': result['metrics']['error_rate'],
            'elapsed_time': result['elapsed_time']
        })

summary_file = output_dir / f"comparison_{timestamp}.json"
with open(summary_file, 'w') as f:
    json.dump(comparison, f, indent=2)

print(f"\nComparison summary: {summary_file}")